<a href="https://colab.research.google.com/github/Lucio2108/ENTREGA-2/blob/main/entrega_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import pandas as pd
import numpy as np
import re

# 1. Leer el dataset original de autos subido a Colab
df_autos = pd.read_csv('Cars Datasets 2025 (1).csv', encoding='latin1')

# Función para limpiar el texto de precios (ej. "$1,100,000" -> 1100000.0)
def extraer_precio(valor):
    if pd.isna(valor):
        return np.nan
    coincidencias = re.findall(r'[\d,]+', str(valor))
    for m in coincidencias:
        limpio = m.replace(',', '')
        if limpio.isdigit() and len(limpio) >= 4:
            return float(limpio)
    return np.nan

# 2. Armar el Maestro de Productos (Catálogo en Excel)
df_autos['id_producto'] = range(1, len(df_autos) + 1)
df_autos['nombre_producto'] = (
    df_autos['Company Names'].astype(str).str.strip() + ' ' +
    df_autos['Cars Names'].astype(str).str.strip()
)
df_autos['precio_unitario'] = df_autos['Cars Prices'].apply(extraer_precio)

df_catalogo = df_autos[[
    'id_producto', 'nombre_producto', 'precio_unitario',
    'Engines', 'HorsePower', 'Fuel Types'
]].dropna(subset=['precio_unitario']).copy()

df_catalogo.to_excel('catalogo_productos.xlsx', index=False, engine='openpyxl')

# 3. Generar el CSV de Transacciones (incluyendo nulos e IDs desprolijos para limpiar después)
np.random.seed(42)
ids_validos = df_catalogo['id_producto'].sample(50, replace=True, random_state=42).astype(str).tolist()
ids_validos[3] = " 15 "
ids_validos[10] = None

fechas = pd.date_range(start='2026-01-05', periods=50, freq='D').strftime('%Y-%m-%d').tolist()
fechas[22] = None

cantidades = np.random.randint(1, 5, size=50).astype(float).tolist()
cantidades[7] = np.nan
cantidades[18] = np.nan

df_transacciones = pd.DataFrame({
    'id_transaccion': range(5001, 5051),
    'id_producto': ids_validos,
    'cantidad': cantidades,
    'fecha': fechas
})

df_transacciones.to_csv('transacciones.csv', index=False)
print("✅ Archivos 'catalogo_productos.xlsx' y 'transacciones.csv' generados con éxito.")

✅ Archivos 'catalogo_productos.xlsx' y 'transacciones.csv' generados con éxito.


In [4]:
# ==========================================
# 1. ADQUISICIÓN DE DATOS
# ==========================================
df_transacciones = pd.read_csv('transacciones.csv')
df_productos = pd.read_excel('catalogo_productos.xlsx', engine='openpyxl')

print(f"Transacciones iniciales: {len(df_transacciones)} filas")
print(f"Catálogo de autos: {len(df_productos)} modelos")

# ==========================================
# 2. TRANSFORMACIÓN Y LIMPIEZA
# ==========================================
# Eliminar registros sin llave de producto o sin fecha
df_transacciones = df_transacciones.dropna(subset=['id_producto', 'fecha']).copy()

# Imputar nulos en cantidad (1 unidad por defecto)
df_transacciones['cantidad'] = df_transacciones['cantidad'].fillna(1)

# Normalizar tipos de datos para que el merge no falle (ambos id_producto como int)
df_transacciones['id_producto'] = (
    df_transacciones['id_producto'].astype(str).str.strip().astype(float).astype(int)
)
df_productos['id_producto'] = df_productos['id_producto'].astype(int)

# Convertir cantidad a entero y fecha a datetime
df_transacciones['cantidad'] = df_transacciones['cantidad'].astype(int)
df_transacciones['fecha'] = pd.to_datetime(df_transacciones['fecha'], errors='coerce')
df_transacciones = df_transacciones.dropna(subset=['fecha'])

# ==========================================
# 3. INTEGRACIÓN (MERGE) Y COLUMNA CALCULADA
# ==========================================
df_consolidado = pd.merge(
    df_transacciones,
    df_productos,
    on='id_producto',
    how='inner'
)

# Crear columna calculada 'total_venta'
df_consolidado['total_venta'] = df_consolidado['cantidad'] * df_consolidado['precio_unitario']
df_consolidado = df_consolidado.sort_values(by='fecha').reset_index(drop=True)

print("\n--- Verificación de valores nulos finales ---")
print(df_consolidado.isnull().sum())

# ==========================================
# 4. EXPORTACIÓN A PARQUET
# ==========================================
df_consolidado.to_parquet('ventas_consolidadas.parquet', engine='pyarrow', index=False)
print("\n✅ Archivo 'ventas_consolidadas.parquet' exportado correctamente.")

# Mostrar las primeras 10 filas como tabla interactiva en Colab
df_consolidado.head(10)

Transacciones iniciales: 50 filas
Catálogo de autos: 1217 modelos

--- Verificación de valores nulos finales ---
id_transaccion     0
id_producto        0
cantidad           0
fecha              0
nombre_producto    0
precio_unitario    0
Engines            0
HorsePower         0
Fuel Types         0
total_venta        0
dtype: int64

✅ Archivo 'ventas_consolidadas.parquet' exportado correctamente.


,id_transaccion,id_producto,cantidad,fecha,nombre_producto,precio_unitario,Engines,HorsePower,Fuel Types,total_venta
0,5001,1127,3,2026-01-05,Mitsubishi Minicab MiEV,18000,Electric Motor,64 hp,Electric,54000
1,5002,861,4,2026-01-06,Kia Rio S,17500,1.6L 4-Cylinder,120 hp,Petrol,70000
2,5003,1131,1,2026-01-07,Mazda CX-3,21000,2.0L SkyActiv-G,146 hp,Petrol,21000
3,5004,15,3,2026-01-08,FERRARI F8 SPIDER,280000,V8,710 hp,Petrol,840000
4,5005,1045,3,2026-01-09,Ford Everest,52000,Turbocharged Inline-4,210 hp,Diesel,156000
5,5006,122,4,2026-01-10,ROLLS ROYCE CULLINAN RED DIAMOND,421000,V12,563 hp,Petrol,1684000
6,5007,467,1,2026-01-11,Nissan GT-R LM NISMO,600000,3.0L Twin-Turbo V6 Hybrid,500 HP,Hybrid,600000
7,5008,331,1,2026-01-12,Volkswagen Polo TDI,20000,1.4L I4 Diesel / 1.6L Diesel,75 - 105 hp,Diesel,20000
8,5009,88,3,2026-01-13,MERCEDES BENZ GLE 450,63000,V6,362 hp,Petrol,189000
9,5010,1124,2,2026-01-14,Mitsubishi Grand Lancer,20000,1.8L MIVEC,138 hp,Petrol,40000
